# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
print("""
Unit of Analysis

One row represents the daily search and analytics performance of one content item
(content_hash_id) for one client (client_hash_id) on one report_date.

Time Window

For this assignment I will analyse a single monthly slice (for example March 2026)
to validate the data contract before building ML features.
""")


Unit of Analysis

One row represents the daily search and analytics performance of one content item
(content_hash_id) for one client (client_hash_id) on one report_date.

Time Window

For this assignment I will analyse a single monthly slice (for example March 2026)
to validate the data contract before building ML features.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
print("""
FEATURES
--------
gsc_impressions
gsc_clicks
gsc_avg_position
ga4_pageviews
ga4_sessions

These values are available before making a refresh recommendation.

LABEL / PROXY
-------------
Future content performance after refresh.
For later assignments this will be represented by a defined proxy rather than used as a feature.

CONTEXT
-------
report_date
client_hash_id
content_hash_id
content_type
content_created_date

These identify the observation or provide grouping information but should not be model features.

EXCLUDED
--------
keyword_hash_id
url_hash_id

Reason:
These are identifiers only.

ga4_data_available
client_has_ga4
client_has_gsc

Reason:
Used only for filtering valid rows.

Future information after the prediction date is excluded to prevent data leakage.
""")


FEATURES
--------
gsc_impressions
gsc_clicks
gsc_avg_position
ga4_pageviews
ga4_sessions

These values are available before making a refresh recommendation.

LABEL / PROXY
-------------
Future content performance after refresh.
For later assignments this will be represented by a defined proxy rather than used as a feature.

CONTEXT
-------
report_date
client_hash_id
content_hash_id
content_type
content_created_date

These identify the observation or provide grouping information but should not be model features.

EXCLUDED
--------
keyword_hash_id
url_hash_id

Reason:
These are identifiers only.

ga4_data_available
client_has_ga4
client_has_gsc

Reason:
Used only for filtering valid rows.

Future information after the prediction date is excluded to prevent data leakage.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
from google.colab import userdata

HF_Token = userdata.get("HF_Token")

print("Token loaded successfully!")

Token loaded successfully!


In [12]:
from datasets import load_dataset
import pandas as pd

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_Token,      # <-- use HF_TOKEN
    streaming=True
)

rows = []

for i, row in enumerate(daily["train"]):
    rows.append(row)
    if i == 999:
        break

df = pd.DataFrame(rows)

print(df.shape)

display(df.head())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(1000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [13]:
print("Earliest date:", df["report_date"].min())
print("Latest date:", df["report_date"].max())

filtered = df[df["ga4_data_available"] == True]

print("Rows with GA4 available:", len(filtered))

filtered = df[df["gsc_data_available"] == True]

print("Rows with GSC available:", len(filtered))

print(df.isnull().sum())

Earliest date: 2025-01-27
Latest date: 2025-01-30
Rows with GA4 available: 0
Rows with GSC available: 1000
report_date                 0
client_hash_id              0
content_hash_id             0
client_has_gsc              0
client_has_ga4              0
gsc_data_available          0
ga4_data_available          0
gsc_impressions             0
gsc_clicks                  0
gsc_sum_position            0
gsc_avg_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
sessions_organic            0
sessions_direct             0
sessions_referral           0
sessions_social             0
sessions_paid               0
sessions_ai                 0
ai_chatgpt                  0
ai_perplexity               0
ai_gemini                   0
ai_copilot                  0
ai_claude                   0
ai_meta                     0
ai_other                    0
scroll_events          

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
print("""
Data Limits

This dataset contains historical search and analytics observations.

It cannot prove why search performance changed.

GA4 metrics are unavailable for some clients before GA4 collection started, so
ga4_data_available must be checked before analysis.

The warehouse contains observational data only, therefore all findings should
be interpreted as decision-support rather than causal conclusions.
""")


Data Limits

This dataset contains historical search and analytics observations.

It cannot prove why search performance changed.

GA4 metrics are unavailable for some clients before GA4 collection started, so
ga4_data_available must be checked before analysis.

The warehouse contains observational data only, therefore all findings should
be interpreted as decision-support rather than causal conclusions.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.